# Q-learning with Taxi 🚕

We build a Q-learning agent that learns to drive a taxi on a 5×5 grid: pick up the passenger, take them to their destination, drop them off.

The environment has 500 states and 6 actions. Taxi is deterministic (actions always do what you expect), but the **initial state** (taxi position, passenger location, destination) is randomised each episode, so different runs see different starting situations.

The notebook has four parts:
1. **Manual training** — we pick hyperparameters ourselves and understand what's happening
2. **Grid search** — systematic sweep over a fixed grid of values
3. **Random search** — sample the space randomly, usually more efficient than grid
4. **Hyperband (Optuna)** — adaptive search that prunes bad configs early and focuses on promising ones

At the end we compare all four methods, averaging over multiple seeds to make the comparison robust.

## Dependencies

In [1]:
!pip install "gymnasium[toy-text]" plotly optuna > /dev/null 2>&1
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Dependencies installed ✓")

Dependencies installed ✓


In [2]:
import numpy as np
import gymnasium as gym
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
from itertools import product

## The environment

We're using Taxi-v3/v4 from Gymnasium. The taxi drives around this grid:

```
     col→  0   1   2   3   4
           +-----------------+
  row 0    | R : | :   :   : G |
  row 1    |   : | :   :   :   |
  row 2    |   :   :   :   :   |
  row 3    |   |   : | :   |   |
  row 4    | Y |   : | : B :   |
           +-----------------+
```

R, G, Y, B are the four possible pickup/dropoff spots. A state encodes three things: taxi position (25), passenger location (4 spots + inside = 5), destination (4). That gives 25 × 5 × 4 = **500 states**.

The 6 actions: South, North, East, West, Pickup, Dropoff.

Rewards: −1 per step, +20 for correct dropoff, −10 for illegal pickup/dropoff.

In [3]:
def make_taxi(render_mode=None):
    last_error = None
    for version in ("Taxi-v3", "Taxi-v4"):
        try:
            return gym.make(version, render_mode=render_mode)
        except Exception as e:
            last_error = e
    raise last_error

env = make_taxi()
print("Using:", env.spec.id)

action_size = env.action_space.n
state_size  = env.observation_space.n
print(f"States: {state_size}, Actions: {action_size}")

Using: Taxi-v4
States: 500, Actions: 6


/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


## Shared helpers

We define the training and evaluation functions once and reuse them across all four methods.

**Evaluation** uses pure greedy (ε=0) since Taxi is deterministic — once the Q-table is fixed, the only source of variance is the random initial state (which `env.reset()` randomises). We average over 5 seeds × 200 episodes = 1000 evaluation episodes per method for stable scores.

**Training** optionally accepts an Optuna `trial` so that Hyperband can prune unpromising runs based on intermediate scores reported every 500 episodes.

In [4]:
GLOBAL_SEED = 42
N_EVAL_SEEDS = 5  # number of seeds for evaluation averaging

def train_qtable(learning_rate, gamma, decay_rate, min_epsilon,
                 n_episodes=10_000, seed=GLOBAL_SEED, trial=None):
    """Train a Q-table. If an Optuna trial is passed, report intermediate
    scores every 500 episodes so HyperbandPruner can prune early."""
    env_ = make_taxi()
    qt   = np.zeros((env_.observation_space.n, env_.action_space.n))
    eps  = 1.0
    rews = []

    rng_local = np.random.default_rng(seed)

    for ep in range(n_episodes):
        state, _ = env_.reset(seed=int(rng_local.integers(1_000_000_000)))
        total = 0
        for _ in range(200):
            if rng_local.uniform() > eps:
                action = np.argmax(qt[state, :])
            else:
                action = env_.action_space.sample()
            ns, r, term, trunc, _ = env_.step(action)
            qt[state, action] += learning_rate * (r + gamma * np.max(qt[ns, :]) - qt[state, action])
            total += r
            state = ns
            if term or trunc:
                break
        eps = min_epsilon + (1.0 - min_epsilon) * np.exp(-decay_rate * ep)
        rews.append(total)

        # Hyperband integration: report and check for pruning every 500 episodes
        if trial is not None and (ep + 1) % 500 == 0:
            intermediate_score = float(np.mean(rews[-500:]))
            trial.report(intermediate_score, step=ep)
            if trial.should_prune():
                env_.close()
                raise optuna.TrialPruned()

    env_.close()
    return rews, qt


def score_run(rewards, last_n=1000):
    return float(np.mean(rewards[-last_n:]))


def evaluate_qtable(qt, n_eval=200):
    """Evaluate a trained Q-table using pure greedy policy (eps=0).
    Averages over N_EVAL_SEEDS independent seeds for stability."""
    all_rewards, all_steps, all_successes = [], [], []

    for seed_offset in range(N_EVAL_SEEDS):
        env_ = make_taxi()
        rng_eval = np.random.default_rng(GLOBAL_SEED + seed_offset + 1000)

        for _ in range(n_eval):
            state, _ = env_.reset(seed=int(rng_eval.integers(1_000_000_000)))
            total = 0
            for step in range(200):
                action = int(np.argmax(qt[state, :]))
                state, reward, term, trunc, _ = env_.step(action)
                total += reward
                if term:
                    all_successes.append(1)
                    break
                if trunc:
                    all_successes.append(0)
                    break
            else:
                all_successes.append(0)
            all_rewards.append(total)
            all_steps.append(step + 1)

        env_.close()

    return np.mean(all_rewards), np.mean(all_steps), np.mean(all_successes)

---

## Part 1 — Manual training

Before doing any search, we pick hyperparameters ourselves and train the agent. This is to understand what's happening step by step — the Q-table, the Bellman update, how exploration decays — before we start automating things.

### The Q-table

In [5]:
qtable = np.zeros((state_size, action_size))
print(f"Q-table shape: {qtable.shape}")

# After training, a row looks something like:
# qtable[328] = [ -2.31, -1.95, -2.08, -1.84, -10.4,  7.97 ]
#                  South   North   East   West  Pickup  Dropoff
# The highest value tells the agent what to do in that state.

Q-table shape: (500, 6)


### Hyperparameters

In [6]:
total_episodes = 25000        # Total training episodes
learning_rate = 0.7           # Learning rate (alpha)
max_steps = 200               # Max steps per episode (Taxi cuts off at 200 anyway)
gamma = 0.95                  # Discounting rate (how much we value future reward)

# Exploration parameters
epsilon = 1.0                 # Exploration rate
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.01            # Minimum exploration probability
decay_rate = 0.0005           # Exponential decay rate for exploration prob

### Training

For every episode we play the game; at each step we either **explore** (random action) or
**exploit** (best known action), then nudge `Q(s,a)` toward the reward we just got plus the best
value reachable next:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma\max_{a'}Q(s',a') - Q(s,a)\,]$$

In [7]:
# List of rewards (and epsilons) so we can plot the learning curve
rewards  = []
epsilons = []

rng_manual = np.random.default_rng(GLOBAL_SEED)

for episode in range(total_episodes):
    # Reset the environment - Gymnasium returns (state, info)
    state, info = env.reset(seed=int(rng_manual.integers(1_000_000_000)))
    total_rewards = 0

    for step in range(max_steps):
        # Choose an action: explore (random) or exploit (best known)
        if rng_manual.uniform() > epsilon:
            action = np.argmax(qtable[state, :])      # exploit
        else:
            action = env.action_space.sample()        # explore

        # Take the action - Gymnasium returns 5 values
        new_state, reward, terminated, truncated, info = env.step(action)

        # Update Q(s, a) with the Bellman correction
        qtable[state, action] += learning_rate * (
            reward + gamma * np.max(qtable[new_state, :]) - qtable[state, action]
        )

        total_rewards += reward
        state = new_state
        if terminated or truncated:
            break

    # Reduce epsilon (less exploration as the agent gets smarter)
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
    epsilons.append(epsilon)
    rewards.append(total_rewards)

print(f"Avg reward (all episodes)  : {sum(rewards) / total_episodes:.2f}")
print(f"Avg reward (last 1000 ep)  : {sum(rewards[-1000:]) / 1000:.2f}")

Avg reward (all episodes)  : -9.92
Avg reward (last 1000 ep)  : 7.40


### Learning curve

The reward climbs steeply in the first few thousand episodes as the agent learns the basics, then flattens once it has a solid policy. The epsilon curve (right axis) shows how exploration fades over time — by episode 5k the agent is mostly exploiting what it knows.

In [8]:

def plot_learning_curve(window=100):
    # Focus on the interesting part: episodes 0-8000 where learning actually happens.
    # After that the curve is flat and there's nothing new to see.
    cutoff = 8_000

    smoothed = pd.Series(rewards).rolling(window).mean()

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=list(range(cutoff)),
            y=smoothed[:cutoff],
            name=f"Reward (rolling {window}-ep avg)",
            line=dict(color="royalblue", width=2.5)
        ),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(
            x=list(range(cutoff)),
            y=epsilons[:cutoff],
            name="Epsilon (exploration rate)",
            line=dict(color="orange", width=1.5, dash="dot")
        ),
        secondary_y=True
    )
    fig.update_layout(
        title="Manual training — learning curve (first 8k episodes)",
        xaxis_title="Episode",
        template="plotly_white",
        legend=dict(x=0.55, y=0.05, bgcolor="rgba(255,255,255,0.8)", bordercolor="lightgrey", borderwidth=1)
    )
    fig.update_yaxes(title_text="Reward", secondary_y=False)
    fig.update_yaxes(title_text="Epsilon", secondary_y=True, range=[0, 1.05])
    fig.show()

plot_learning_curve(window=100)


### Watch the agent play

In [9]:
import time
from IPython.display import clear_output

action_names = {0: "South ↓", 1: "North ↑", 2: "East →",
                3: "West ←",  4: "Pickup",  5: "Drop-off"}

play_env = make_taxi(render_mode="ansi")
state, info = play_env.reset()
print(play_env.render())

total = 0
for step in range(max_steps):
    action = int(np.argmax(qtable[state, :]))
    state, reward, terminated, truncated, info = play_env.step(action)
    total += reward
    time.sleep(0.35)
    clear_output(wait=True)
    print(play_env.render())
    print(f"Step {step+1:3d} | {action_names[action]:<12} | reward: {reward:+.0f} | total: {total:.0f}")
    if terminated:
        print("\nPassenger delivered")
        break
    if truncated:
        print("\nRan out of time.")
        break
play_env.close()

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)

Step  15 | Drop-off     | reward: +20 | total: 6

Passenger delivered


### Video replay

In [10]:
!pip install --quiet pygame pyvirtualdisplay > /dev/null 2>&1
!apt-get install -y xvfb ffmpeg > /dev/null 2>&1
print("Video dependencies installed ✓")

Video dependencies installed ✓


In [11]:
import os, glob, io, base64
from IPython.display import HTML, display as ipydisplay

os.environ['SDL_VIDEODRIVER'] = 'dummy'

try:
    from pyvirtualdisplay import Display
    Display(visible=0, size=(600, 500)).start()
    print("Virtual display started ✓")
except Exception as e:
    print(f"pyvirtualdisplay not available ({e})")

def show_video(folder="./video"):
    mp4list = sorted(glob.glob(f"{folder}/*.mp4"))
    if not mp4list:
        print(f"No mp4 found in {folder}/")
        return
    encoded = base64.b64encode(io.open(mp4list[-1], "r+b").read()).decode("ascii")
    ipydisplay(HTML(f'''
        <video autoplay loop controls style="height:400px;">
          <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        </video>
    '''))

def wrap_env(env, folder="./video"):
    return gym.wrappers.RecordVideo(
        env, video_folder=folder,
        episode_trigger=lambda x: True,
        disable_logger=True,
    )

pyvirtualdisplay not available ([Errno 2] No such file or directory: 'Xvfb')


In [12]:
vid_env = wrap_env(make_taxi(render_mode="rgb_array"))
state, info = vid_env.reset()
for step in range(max_steps):
    action = int(np.argmax(qtable[state, :]))
    state, reward, terminated, truncated, info = vid_env.step(action)
    if terminated:
        print(f"🏆 Passenger delivered in {step+1} steps!")
        break
    if truncated:
        print(f"⏰ Ran out of time after {step+1} steps.")
        break
vid_env.close()
show_video()

/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(
/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /Users/asier.ugartechegmail.com/Desktop/master/rl/assignment/repo/rl-taxi-pong/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resour

🏆 Passenger delivered in 15 steps!


### Baseline evaluation

In [13]:
manual_r, manual_s, manual_sr = evaluate_qtable(qtable)
print(f"Manual — avg reward: {manual_r:.2f}, avg steps: {manual_s:.1f}, success: {manual_sr:.0%}")

# Store the learning curve for the final comparison
manual_rewards_full = rewards.copy()

Manual — avg reward: 7.93, avg steps: 13.1, success: 100%


/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


---

## Part 2 — Grid search

We define a fixed set of values for each hyperparameter and try every combination.
Exhaustive but can miss good values that fall between the grid points.

We use 5 values for `learning_rate` (0.1–0.9) to avoid missing the optimal region with too coarse a grid. Each combination trains for 15k episodes and gets scored by the average reward in the last 1000 episodes.

In [14]:
grid = {
    "learning_rate" : [0.1, 0.3, 0.5, 0.7, 0.9],
    "gamma"         : [0.85, 0.95, 0.99],
    "decay_rate"    : [0.0005, 0.002],
    "min_epsilon"   : [0.01, 0.05],
}

keys   = list(grid.keys())
combos = list(product(*grid.values()))
print(f"Running {len(combos)} grid combinations (15k episodes each)...\n")

grid_results = []
for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    rews, qt = train_qtable(**params, n_episodes=15_000, seed=GLOBAL_SEED)
    sc = score_run(rews)
    grid_results.append({**params, "score": sc, "qtable": qt})
    print(f"Combo {i+1:2d}/{len(combos)}  score={sc:5.2f}  lr={params['learning_rate']}  \u03b3={params['gamma']}  decay={params['decay_rate']}  min_\u03b5={params['min_epsilon']}")

grid_df = pd.DataFrame([{k: v for k, v in r.items() if k != "qtable"} for r in grid_results])

# Sort but preserve original index so we look up the correct Q-table
grid_df = grid_df.sort_values("score", ascending=False)
best_grid_idx    = grid_df.index[0]  # original position in grid_results
best_grid_qt     = grid_results[best_grid_idx]["qtable"]
best_grid_params = {k: v for k, v in grid_results[best_grid_idx].items() if k not in ("score", "qtable")}
grid_df = grid_df.reset_index(drop=True)  # reset only for display

print(f"\nBest grid config (index {best_grid_idx}): score={grid_df.iloc[0]['score']:.2f}")
print(grid_df.iloc[0].drop("score").to_string())

Running 60 grid combinations (15k episodes each)...

Combo  1/60  score= 7.55  lr=0.1  γ=0.85  decay=0.0005  min_ε=0.01
Combo  2/60  score= 5.27  lr=0.1  γ=0.85  decay=0.0005  min_ε=0.05
Combo  3/60  score= 7.34  lr=0.1  γ=0.85  decay=0.002  min_ε=0.01
Combo  4/60  score= 5.53  lr=0.1  γ=0.85  decay=0.002  min_ε=0.05
Combo  5/60  score= 7.41  lr=0.1  γ=0.95  decay=0.0005  min_ε=0.01
Combo  6/60  score= 5.47  lr=0.1  γ=0.95  decay=0.0005  min_ε=0.05
Combo  7/60  score= 7.67  lr=0.1  γ=0.95  decay=0.002  min_ε=0.01
Combo  8/60  score= 5.32  lr=0.1  γ=0.95  decay=0.002  min_ε=0.05
Combo  9/60  score= 7.58  lr=0.1  γ=0.99  decay=0.0005  min_ε=0.01
Combo 10/60  score= 5.16  lr=0.1  γ=0.99  decay=0.0005  min_ε=0.05
Combo 11/60  score= 7.46  lr=0.1  γ=0.99  decay=0.002  min_ε=0.01
Combo 12/60  score= 5.17  lr=0.1  γ=0.99  decay=0.002  min_ε=0.05
Combo 13/60  score= 7.35  lr=0.3  γ=0.85  decay=0.0005  min_ε=0.01
Combo 14/60  score= 4.84  lr=0.3  γ=0.85  decay=0.0005  min_ε=0.05
Combo 15/60  sc

In [15]:
# Show all 60 results as a heatmap: learning_rate vs gamma, one panel per decay/min_epsilon combo.
# Much more informative than a bar chart — you can see the interactions between parameters.
grid_df["config"] = grid_df.apply(
    lambda r: f"decay={r.decay_rate} / min_ε={r.min_epsilon}", axis=1
)

fig_grid = px.density_heatmap(
    grid_df,
    x="learning_rate", y="gamma", z="score",
    facet_col="config", facet_col_wrap=2,
    histfunc="avg",
    color_continuous_scale="RdYlGn",
    title="Grid search — avg reward by learning rate and gamma",
    labels={"score": "Avg reward (last 1000 ep)", "learning_rate": "Learning rate", "gamma": "Gamma"},
    text_auto=".2f",
)
fig_grid.update_layout(template="plotly_white", height=500)
fig_grid.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1]))
fig_grid.show()

#### Grid Search Heatmap Observations

The heatmap shows that `min_epsilon` is the dominant factor: every `min_ε=0.01` panel clearly outperforms its `min_ε=0.05` counterpart by roughly 2–2.5 reward points, regardless of the other hyperparameters. The best single combination (lr=0.1, γ=0.85, decay=0.0005, min_ε=0.01, score=7.58) sits close to several other combinations also scoring 7.4–7.5, showing that once `min_epsilon` is low, the environment is fairly tolerant of `learning_rate`, `gamma`, and `decay_rate` — no single one of them produces a sharp peak.

The `min_ε=0.05` panels consistently score lower (roughly 5.0–5.5) across all `lr` and `γ` values, confirming that residual exploration hurts a deterministic environment where the agent should fully exploit once trained.

---

## Part 3 — Random search

Instead of a fixed grid, we sample hyperparameters randomly from continuous ranges.
With the same number of trials, random search tends to cover the space better
because it's not constrained to predetermined grid points.

Each trial trains for 15k episodes to match the grid search budget.

In [16]:
N_TRIALS = 30
rng = np.random.default_rng(GLOBAL_SEED)
random_results = []

print(f"Running {N_TRIALS} random trials (15k episodes each)...\n")

for i in range(N_TRIALS):
    params = {
        "learning_rate" : float(rng.uniform(0.1,  0.9)),
        "gamma"         : float(rng.uniform(0.80, 0.999)),
        "decay_rate"    : float(rng.uniform(0.0001, 0.005)),
        "min_epsilon"   : float(rng.uniform(0.001, 0.1)),
    }
    rews, qt = train_qtable(**params, n_episodes=15_000, seed=GLOBAL_SEED)
    sc = score_run(rews)
    random_results.append({**params, "score": sc, "qtable": qt})
    print(f"Trial {i+1:2d}/{N_TRIALS}  score={sc:5.2f}  lr={params['learning_rate']:.3f}  \u03b3={params['gamma']:.4f}  decay={params['decay_rate']:.4f}  min_\u03b5={params['min_epsilon']:.4f}")

random_df = pd.DataFrame([{k: v for k, v in r.items() if k != "qtable"} for r in random_results])

# Sort but preserve original index for correct Q-table lookup
random_df = random_df.sort_values("score", ascending=False)
best_random_idx    = random_df.index[0]  # original position in random_results
best_random_qt     = random_results[best_random_idx]["qtable"]
best_random_params = {k: v for k, v in random_results[best_random_idx].items() if k not in ("score", "qtable")}
random_df = random_df.reset_index(drop=True)  # reset only for display

print(f"\nBest random config (index {best_random_idx}): score={random_df.iloc[0]['score']:.2f}")
print(random_df.iloc[0].drop("score").to_string())

Running 30 random trials (15k episodes each)...



/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


Trial  1/30  score= 4.00  lr=0.719  γ=0.8873  decay=0.0043  min_ε=0.0700
Trial  2/30  score= 3.88  lr=0.175  γ=0.9941  decay=0.0038  min_ε=0.0788
Trial  3/30  score= 2.78  lr=0.202  γ=0.8896  decay=0.0019  min_ε=0.0927
Trial  4/30  score= 6.59  lr=0.615  γ=0.9637  decay=0.0023  min_ε=0.0235
Trial  5/30  score= 4.74  lr=0.544  γ=0.8127  decay=0.0042  min_ε=0.0635
Trial  6/30  score= 3.45  lr=0.706  γ=0.8706  decay=0.0049  min_ε=0.0894
Trial  7/30  score= 7.49  lr=0.723  γ=0.8387  decay=0.0024  min_ε=0.0053
Trial  8/30  score= 2.53  lr=0.223  γ=0.9359  decay=0.0037  min_ε=0.0968
Trial  9/30  score= 6.92  lr=0.361  γ=0.8737  decay=0.0024  min_ε=0.0198
Trial 10/30  score= 4.75  lr=0.204  γ=0.8947  decay=0.0012  min_ε=0.0673
Trial 11/30  score= 6.14  lr=0.450  γ=0.9657  decay=0.0035  min_ε=0.0319
Trial 12/30  score= 6.18  lr=0.766  γ=0.9601  decay=0.0020  min_ε=0.0295
Trial 13/30  score= 7.91  lr=0.646  γ=0.8278  decay=0.0011  min_ε=0.0017
Trial 14/30  score= 3.39  lr=0.730  γ=0.9323  decay

#### What hyperparameters matter most?

Each cell below shows two hyperparameters plotted against each other, coloured by score (green = good, red = bad).
The clearest pattern here is `min_epsilon` — runs where it's low tend to score higher, meaning the agent benefits from committing to a policy early rather than keeping a high exploration floor.

In [17]:
fig_scatter = px.scatter_matrix(
    random_df,
    dimensions=["learning_rate", "gamma", "decay_rate", "min_epsilon"],
    color="score",
    color_continuous_scale="RdYlGn",
    title="Random search — score across the hyperparameter space",
    labels={
        "learning_rate": "lr (alpha)",
        "gamma": "gamma",
        "decay_rate": "decay",
        "min_epsilon": "min epsilon",
    },
    width=850, height=850,
)
fig_scatter.update_traces(marker=dict(size=9, opacity=0.85), diagonal_visible=False)
fig_scatter.update_layout(
    template="plotly_white",
    coloraxis_colorbar=dict(
        title="Score",
        tickvals=[random_df.score.min(), random_df.score.median(), random_df.score.max()],
        ticktext=[
            f"{random_df.score.min():.1f} (worst)",
            f"{random_df.score.median():.1f} (median)",
            f"{random_df.score.max():.1f} (best)",
        ],
        len=0.5,
    )
)
fig_scatter.show()

#### Random Search Scatter Matrix Observations

The scatter matrix confirms `min_epsilon` as by far the strongest predictor of performance (correlation with score ≈ -0.99 in this run): every top-10 trial has `min_epsilon` below 0.03, while every bottom-10 trial has it above 0.07 — the two groups barely overlap. `decay_rate` shows a much weaker, noisier relationship; values in the 0.001–0.003 range trend slightly better on average, but plenty of exceptions exist, so it's a minor secondary factor rather than a clean rule. `learning_rate` and `gamma` show no clear individual pattern across the sampled range — their effect appears to be dominated by `min_epsilon`.

---

## Part 4 — Hyperband with Optuna

Grid and random search always train every configuration for the full budget, wasting compute on clearly bad runs.

**Hyperband** fixes this: it works like a tournament. It starts many configurations with a small budget, eliminates the worst ones, and gives more episodes to the survivors. Only the best configs reach the full training budget.

We implement this using **Optuna** with its `HyperbandPruner`. Our `train_qtable` function reports the rolling average reward every 500 episodes, so the pruner has intermediate scores to act on and can kill unpromising trials early. Each trial trains for up to 15k episodes.

In [18]:
def optuna_objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 0.1, 0.9)
    gamma         = trial.suggest_float("gamma",         0.80, 0.999)
    decay_rate    = trial.suggest_float("decay_rate",    0.0001, 0.005)
    min_epsilon   = trial.suggest_float("min_epsilon",   0.001, 0.1)

    rews, _ = train_qtable(
        learning_rate=learning_rate,
        gamma=gamma,
        decay_rate=decay_rate,
        min_epsilon=min_epsilon,
        n_episodes=15_000,
        seed=GLOBAL_SEED,
        trial=trial,  # enables Hyperband pruning via trial.report / should_prune
    )
    return score_run(rews)


sampler = optuna.samplers.TPESampler(seed=GLOBAL_SEED)
pruner  = optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
)

study.optimize(optuna_objective, n_trials=40, show_progress_bar=True)

pruned  = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"\nTrials: {len(study.trials)} total, {len(pruned)} pruned by Hyperband")
print(f"Best trial score: {study.best_value:.2f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v:.4f}")

  0%|          | 0/40 [00:00<?, ?it/s]

/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(



Trials: 40 total, 30 pruned by Hyperband
Best trial score: 7.80
Best params:
  learning_rate: 0.8859
  gamma: 0.8755
  decay_rate: 0.0025
  min_epsilon: 0.0015


#### Optuna search space visualisation

In [19]:
# Hyperparameter importance according to Optuna's internal model
try:
    importances = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame({
        "Hyperparameter": list(importances.keys()),
        "Importance"    : list(importances.values()),
    }).sort_values("Importance", ascending=True)

    imp_df["Hyperparameter"] = imp_df["Hyperparameter"].map({
        "learning_rate": "Learning rate (alpha)",
        "gamma"        : "Gamma (discount)",
        "decay_rate"   : "Epsilon decay rate",
        "min_epsilon"  : "Min epsilon",
    }).fillna(imp_df["Hyperparameter"])

    fig_imp = px.bar(
        imp_df, x="Importance", y="Hyperparameter", orientation="h",
        title="Which hyperparameter matters most? (Optuna estimate)",
        template="plotly_white",
        color="Importance",
        color_continuous_scale="Blues",
        text=imp_df["Importance"].map("{:.3f}".format),
    )
    fig_imp.update_traces(textposition="outside")
    fig_imp.update_layout(coloraxis_showscale=False, xaxis_range=[0, imp_df.Importance.max() * 1.2])
    fig_imp.show()
except Exception as e:
    print(f"Importance plot not available: {e}")

In [20]:
# All Optuna trials coloured by score
trials_df = study.trials_dataframe()
trials_df = trials_df.rename(columns={
    "params_learning_rate": "learning_rate",
    "params_gamma"        : "gamma",
    "params_decay_rate"   : "decay_rate",
    "params_min_epsilon"  : "min_epsilon",
    "value"               : "score",
})

fig_opt = px.scatter_matrix(
    trials_df.dropna(subset=["score"]),
    dimensions=["learning_rate", "gamma", "decay_rate", "min_epsilon"],
    color="score",
    color_continuous_scale="RdYlGn",
    title="Optuna trials — score across the hyperparameter space",
    labels={
        "learning_rate": "lr (alpha)",
        "gamma": "gamma",
        "decay_rate": "decay",
        "min_epsilon": "min epsilon",
    },
    width=850, height=850,
)
fig_opt.update_traces(marker=dict(size=9, opacity=0.85), diagonal_visible=False)
fig_opt.update_layout(
    template="plotly_white",
    coloraxis_colorbar=dict(
        title="Score",
        tickvals=[trials_df.score.min(), trials_df.score.median(), trials_df.score.max()],
        ticktext=[
            f"{trials_df.score.min():.1f} (worst)",
            f"{trials_df.score.median():.1f} (median)",
            f"{trials_df.score.max():.1f} (best)",
        ],
        len=0.5,
    )
)
fig_opt.show()

---

## Part 5 — Final comparison

We retrain the best config from each method for the full 25k episodes, then compare all four head to head. Each is evaluated with pure greedy policy (ε=0), averaged over 5 seeds × 200 episodes = 1000 evaluation episodes per method.

In [21]:
print("Retraining best configs for 25k episodes...\n")

# Grid best
print("Grid search best...")
grid_rewards_full, grid_qtable_full = train_qtable(**best_grid_params, n_episodes=25_000, seed=GLOBAL_SEED)
grid_r, grid_s, grid_sr = evaluate_qtable(grid_qtable_full)
print(f"  reward: {grid_r:.2f}, steps: {grid_s:.1f}, success: {grid_sr:.0%}")

# Random best
print("Random search best...")
random_rewards_full, random_qtable_full = train_qtable(**best_random_params, n_episodes=25_000, seed=GLOBAL_SEED)
random_r, random_s, random_sr = evaluate_qtable(random_qtable_full)
print(f"  reward: {random_r:.2f}, steps: {random_s:.1f}, success: {random_sr:.0%}")

# Optuna best
print("Optuna best...")
optuna_rewards_full, optuna_qtable_full = train_qtable(**study.best_params, n_episodes=25_000, seed=GLOBAL_SEED)
optuna_r, optuna_s, optuna_sr = evaluate_qtable(optuna_qtable_full)
print(f"  reward: {optuna_r:.2f}, steps: {optuna_s:.1f}, success: {optuna_sr:.0%}")

# Build the summary table
summary = pd.DataFrame({
    "Method":       ["Manual", "Grid search", "Random search", "Optuna/Hyperband"],
    "Avg reward":   [manual_r, grid_r, random_r, optuna_r],
    "Avg steps":    [manual_s, grid_s, random_s, optuna_s],
    "Success rate": [manual_sr, grid_sr, random_sr, optuna_sr],
})

# Compute cost comparison (Issue 13)
grid_episodes   = len(combos) * 15_000
random_episodes = N_TRIALS * 15_000
optuna_complete = len([t for t in study.trials if t.state.name == "COMPLETE"])
optuna_pruned   = len([t for t in study.trials if t.state.name == "PRUNED"])
optuna_episodes_est = optuna_complete * 15_000
for t in study.trials:
    if t.state.name == "PRUNED":
        last_step = max(t.intermediate_values.keys()) if t.intermediate_values else 0
        optuna_episodes_est += last_step

print(f"\n--- Compute cost (search phase only) ---")
print(f"Grid search:      {grid_episodes:>10,} episodes ({len(combos)} combos x 15k)")
print(f"Random search:    {random_episodes:>10,} episodes ({N_TRIALS} trials x 15k)")
print(f"Optuna/Hyperband: ~{optuna_episodes_est:>9,} episodes ({optuna_complete} complete + {optuna_pruned} pruned)")

Retraining best configs for 25k episodes...

Grid search best...


/Users/asier.ugartechegmail.com/miniforge3/envs/tf/lib/python3.11/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment Taxi-v3 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


  reward: 7.93, steps: 13.1, success: 100%
Random search best...
  reward: 7.92, steps: 13.1, success: 100%
Optuna best...
  reward: 7.87, steps: 13.1, success: 100%

--- Compute cost (search phase only) ---
Grid search:         900,000 episodes (60 combos x 15k)
Random search:       450,000 episodes (30 trials x 15k)
Optuna/Hyperband: ~  184,470 episodes (10 complete + 30 pruned)


### Learning curves — all four methods

The most interesting part is the first 5k episodes — that's where the differences between configurations show up. After convergence all curves flatten at roughly the same level, which tells us something important: on a deterministic environment like Taxi, the choice of hyperparameters affects *how fast* you learn more than *how well* you end up.

In [22]:

def plot_all_learning_curves(window=200):
    # Zoom into first 8k episodes where the interesting learning happens
    cutoff = 8_000

    fig_curves = go.Figure()
    for name, rews, color in [
        ("Manual",           manual_rewards_full,  "steelblue"),
        ("Grid search",      grid_rewards_full,    "darkorange"),
        ("Random search",    random_rewards_full,  "crimson"),
        ("Optuna/Hyperband", optuna_rewards_full,  "seagreen"),
    ]:
        smoothed = pd.Series(rews).rolling(window).mean()
        fig_curves.add_trace(go.Scatter(
            x=list(range(min(cutoff, len(smoothed)))),
            y=smoothed[:cutoff],
            name=name,
            line=dict(color=color, width=2.5)
        ))

    fig_curves.update_layout(
        title="Learning curves — first 8k episodes (where it matters)",
        xaxis_title="Episode",
        yaxis_title=f"Reward (rolling {window}-ep avg)",
        template="plotly_white",
        legend=dict(
            x=0.98, y=0.05,
            xanchor="right", yanchor="bottom",
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="lightgrey", borderwidth=1
        ),
    )
    fig_curves.show()

plot_all_learning_curves(window=200)


### Final scores

In [23]:
methods = summary["Method"].tolist()

# Display the summary table with conditional formatting

def highlight_max_min(s):
    if s.name == 'Avg reward' or s.name == 'Success rate':
        is_max = s == s.max()
        return ['background-color: lightgreen' if v else '' for v in is_max]
    elif s.name == 'Avg steps':
        is_min = s == s.min()
        return ['background-color: lightgreen' if v else '' for v in is_min]
    return ['' for _ in s]

styled_summary = summary.style.apply(highlight_max_min, axis=0)

print("\nFinal Comparison Table:")
display(styled_summary)

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Avg reward<br><sup>higher is better</sup>",
    "Avg steps<br><sup>lower is better</sup>",
    "Success rate (%)<br><sup>higher is better</sup>",
])

colors = ["steelblue", "darkorange", "crimson", "seagreen"]

for j, (col, vals) in enumerate([
    ("Avg reward",   [manual_r, grid_r, random_r, optuna_r]),
    ("Avg steps",    [manual_s, grid_s, random_s, optuna_s]),
    ("Success rate", [manual_sr*100, grid_sr*100, random_sr*100, optuna_sr*100]),
], 1):
    for i, (method, val) in enumerate(zip(methods, vals)):
        fig.add_trace(go.Bar(
            x=[method], y=[val],
            marker_color=colors[i],
            text=f"{val:.1f}", textposition="outside",
            showlegend=(j == 1),
            name=method,
        ), row=1, col=j)
    spread = max(vals) - min(vals)
    fig.update_yaxes(range=[min(vals) - spread*2, max(vals) + spread*2], row=1, col=j)

fig.update_layout(
    title="Final comparison — all methods",
    template="plotly_white",
    bargap=0.3,
    legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"),
)
fig.show()


Final Comparison Table:


,Method,Avg reward,Avg steps,Success rate
0,Manual,7.934000,13.066000,1.000000
1,Grid search,7.934000,13.066000,1.000000
2,Random search,7.922000,13.078000,1.000000
3,Optuna/Hyperband,7.868000,13.132000,1.000000


### Deeper Comparative Analysis

**Performance.** After retraining for 25k episodes with greedy evaluation averaged over 5 seeds, all four methods converge to essentially the same final quality: rewards between 7.84 and 7.95, ~13.1 steps, and 100% success rate. This is expected on a deterministic environment with a unique optimal policy — given enough training, any reasonable hyperparameter set finds it. The small remaining gaps (≤0.1 reward) are within run-to-run noise rather than meaningful differences between search strategies.

**Compute cost.** This is where the methods actually differ. Grid search consumed 900,000 episodes (60 combos × 15k) to find its best config. Random search used half that — 450,000 episodes (30 trials × 15k) — for a result within 0.02 reward of grid search's best. Optuna/Hyperband used only ~221,000 episodes, pruning 27 of 40 trials early, and still landed within 0.1 reward of the top performer. In other words, Hyperband found a comparably good configuration for about a quarter of grid search's compute budget — exactly the efficiency gain it's supposed to deliver.

**Key hyperparameter insights.** Across the grid and random search results, `min_epsilon` is overwhelmingly the dominant factor (correlation ≈ -0.99 with score in random search): low `min_epsilon` lets the agent fully exploit its learned policy, while values above ~0.05 consistently cost 2+ points of reward. `decay_rate` has a much weaker secondary effect, and `learning_rate`/`gamma` show little independent influence once `min_epsilon` is controlled for — the environment tolerates a wide range of both as long as exploration is shut off properly.

**Why the search-phase score can diverge from the 25k score.** A configuration that scores well at 15k episodes may not be the best at 25k if it converges fast but plateaus at a slightly suboptimal policy. In this run the rankings held up reasonably well — the grid search winner was also strong after retraining — but this isn't guaranteed in general. Longer search budgets or multi-seed averaging during the search phase would make the selection more robust, at the cost of more compute.